# 使用 LLM 进行人工智能支持的空调（AC）竞争对手分析

## 练习目标（理念）

在竞争激烈的市场里，品牌官网信息分散、非结构化，客户与企业很难快速对比产品。
本项目构建一个 **AI 驱动** 的小系统：抓取官网文本，再用 **大型语言模型（LLM）** 自动比较领先品牌的空调（AC）。

## 和本课 Week 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `requests` + BeautifulSoup 取正文 |
| Chat Completions API | OpenAI 兼容客户端调本地 Ollama |
| `messages`（system / user） | `call_llm` 里组装角色消息 |
| 业务应用 | 把非结构化网页变成「对比表 + 建议」 |

## 目标

开发一个系统：
- 从公司网站提取产品相关数据
- 识别主要功能、定价与规格线索
- 使用 **LLM** 生成结构化对比

## 怎么跑

1. 本机启动 Ollama，并确保已有 `llama3.2:1b`（或改 `__init__` 里的 `model`）
2. 自上而下运行：先定义 `ACComparator`，再跑抓取与对比那一格
3. 对比结果会 `print` 出来；提示词与 URL 保持英文/原样以便复现



In [ ]:
# ========== 导入与 ACComparator 类：抓取 + 本地 LLM 对比 ==========

# 导入 requests：用 HTTP GET 拉取网页 HTML
import requests
# 从 bs4 导入 BeautifulSoup：解析 HTML、去标签取文本
from bs4 import BeautifulSoup
# 从 urllib.parse 导入 urljoin：拼接相对链接（本格导入后未使用，逻辑保持原样）
from urllib.parse import urljoin

class ACComparator:
    """空调竞品对比器：抓官网正文，再经本地 Ollama（OpenAI 兼容）生成对比。"""

    def __init__(self, model="llama3.2:1b"):
        # 默认本地小模型名：需与 ollama pull 的名字一致；可在构造时覆盖
        self.model = model
        # 请求头：带上简单 User-Agent，降低被网站直接拒绝的概率
        self.headers = {
            "User-Agent": "Mozilla/5.0"
        }

    def smart_fetch(self, url):
        """拉取 URL 的 HTML 文本；失败则返回空字符串（容错，不中断流水线）。"""
        try:
            # timeout=10：最多等 10 秒，避免卡住笔记本
            res = requests.get(url, headers=self.headers, timeout=10)
            # 返回响应正文（HTML 字符串）
            return res.text
        except:
            # 任意异常都吞掉并返回空串（原逻辑如此，便于演示继续往下走）
            return ""
    
    def get_text(self, url):
        """抓取页面并清洗：去掉 script/style，返回纯文本。"""
        # 先拿到原始 HTML
        html = self.smart_fetch(url)
        # 用 html.parser 解析成可遍历的 DOM 树
        soup = BeautifulSoup(html, "html.parser")

        # 删除脚本与样式节点，避免把 JS/CSS 噪声喂给模型
        for tag in soup(["script", "style"]):
            tag.decompose()

        # separator="\n"：块级元素之间换行；strip=True：去掉首尾空白
        return soup.get_text(separator="\n", strip=True)
    
    def call_llm(self, prompt, system_prompt="You are a helpful assistant"):
        """通过 OpenAI 兼容接口调用本地 Ollama，返回助手回复文本。"""
        # 延迟导入 OpenAI 客户端：只在真正调用模型时才加载
        from openai import OpenAI

        # base_url 指向本机 Ollama 的 OpenAI 兼容端点；api_key 对本地常可填占位字符串
        client = OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )

        # Chat Completions：system 定角色，user 放具体对比任务
        response = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ]
        )

        # 取出第一条 choice 的 message.content
        return response.choices[0].message.content
    
    def generate_AC_Comparison(self, name1, data1, name2, data2):
        """把两边品牌名与抓取文本拼进英文 prompt，请模型输出对比与建议。"""
        # f-string 拼 prompt：截断到前 3000 字符，控制上下文长度（发给模型的英文保持原样）
        prompt = f"""
        Compare {name1} and {name2} Air Conditioners.

        DATA {name1}:
        {data1[:3000]}

        DATA {name2}:
        {data2[:3000]}

        Generate:
        1. Comparison table (Price, Features, Energy Efficiency)
        2. Strengths of each
        3. Which is better for Indian users
        4. Final recommendation
        """

        # 交给 call_llm；system 用默认英文助手设定
        return self.call_llm(prompt)



In [ ]:
# ========== 端到端演示：抓取三星 / 松下空调页 → LLM 对比 ==========

# 实例化对比器（默认 model="llama3.2:1b"）
analyst = ACComparator()

# 两个品牌空调落地页 URL（保持原样；站点结构变化可能导致抓取变空）
samsung_url = "https://www.samsung.com/in/air-conditioners/"
panasonic_url = "https://store.in.panasonic.com/air-conditioners.html"

# 分别抓取并清洗为纯文本
samsung_data = analyst.get_text(samsung_url)
panasonic_data = analyst.get_text(panasonic_url)

# 调用 LLM 生成结构化对比（表 + 优劣势 + 建议）
result = analyst.generate_AC_Comparison(
    "Samsung",
    samsung_data,
    "Panasonic",
    panasonic_data
)

# 打印模型返回的对比结果
print(result)



## 结论

该项目展示了 **LLM（大型语言模型）** 如何与网页抓取结合：把非结构化网站文本转化成可执行的业务洞察（对比表、优劣势、购买建议），从而支持更快、更清晰的决策。

复习点：抓取质量决定上下文质量；提示词决定输出结构；本地 Ollama 则让你在无云端费用时也能跑通整条链路。

